Goal: split the data into train, validation, test fairly

# All the required libraries

In [1]:
#Environment
%env OMP_NUM_THREADS=6
import os
os.environ["OMP_NUM_THREADS"] = "6"

# Standards
import ast
import json
import math
import pathlib
import random, hashlib
import re
import shutil
import warnings
from collections import Counter, defaultdict
from datetime import datetime
from decimal import Decimal
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
import folium
from IPython.display import display

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from matplotlib.transforms import Affine2D
from matplotlib.widgets import LassoSelector
import plotly.express as px

# Image
import cv2
from PIL import Image, ImageStat
from easyimages import EasyImage, EasyImageList, bbox
from easyimages.easyimages import CTX


# SciPy 
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from scipy.stats import entropy
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

from pathlib import Path
warnings.simplefilter(action="ignore", category=FutureWarning)
%matplotlib inline

# Data path
rail_data_path = "./rail_data/DB/"

import csv, sys
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Callable



env: OMP_NUM_THREADS=6


# Number of patches per {trackID and side} for all stations(folders) -->  useful for including different tracks in different sets

In [3]:
def extract_frame_number(filename: str):
    """Extract the frame number (the XXX after station name)"""
    name = os.path.splitext(filename)[0]
    # Pattern: station_XXX_timestamp_patches_patch_...
    # Example: 1_calibration_1.2_000_1631441714.100000026_patches_patch_...
    m = re.search(r'_(\d{3})_[\d\.]+_patches_patch', name)
    return int(m.group(1)) if m else None

def extract_station(filename: str):
    name = os.path.splitext(filename)[0]
    m = re.match(r'(.+?\d+\.\d+)', name)
    return m.group(1) if m else None

def extract_label(filename: str):
    name = os.path.splitext(filename)[0]
    m = re.search(r'_(-?\d+)_(left|right)$', name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def extract_side(filename: str):
    name = os.path.splitext(filename)[0]
    m = re.search(r'_(left|right)$', name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else None

def read_folder_with_frames(root, label_name, nice_label):
    """Modified to include frame ordering"""
    folder = os.path.join(root, label_name)
    rows = []
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if not os.path.isfile(path):
            continue
        frame_num = extract_frame_number(f)
        station = extract_station(f)
        track_id = extract_label(f)
        side = extract_side(f)
        
        # Only add if all extractions succeeded
        if all(x is not None for x in [frame_num, station, track_id, side]):
            rows.append({
                "Station": station,
                "TrackID": track_id,
                "Side": side,
                "Class": nice_label,
                "Frame": frame_num,
                "Filename": f
            })
    return pd.DataFrame(rows)

def analyze_temporal_clustering(labeled_root):
    """
    Analyze whether defects cluster in consecutive frames or appear isolated
    """
    # Read both folders with frame information
    df_good = read_folder_with_frames(labeled_root, "sequential_noprep_good", "good")
    df_bad  = read_folder_with_frames(labeled_root, "sequential_noprep_defects_maskscreated", "defective")
    
    df = pd.concat([df_good, df_bad], ignore_index=True)
    
    # Create binary defect indicator
    df['is_defective'] = (df['Class'] == 'defective').astype(int)
    
    results = []
    
    # Group by sequence (station + trackID + side)
    for (station, track_id, side), group in df.groupby(['Station', 'TrackID', 'Side']):
        # Sort by frame number (numeric order)
        group = group.sort_values('Frame').reset_index(drop=True)
        defects = group['is_defective'].values
        frames = group['Frame'].values
        
        n_defective = defects.sum()
        if n_defective == 0:  # Skip sequences with no defects
            continue
        
        # Count isolated vs clustered defects
        isolated = 0
        clustered = 0
        
        for i, is_def in enumerate(defects):
            if is_def:
                has_prev_defect = (i > 0 and defects[i-1])
                has_next_defect = (i < len(defects)-1 and defects[i+1])
                
                if has_prev_defect or has_next_defect:
                    clustered += 1
                else:
                    isolated += 1
        
        # Calculate run lengths for defects
        runs = []
        current_run = 0
        for is_def in defects:
            if is_def:
                current_run += 1
            else:
                if current_run > 0:
                    runs.append(current_run)
                    current_run = 0
        if current_run > 0:  # Don't forget last run
            runs.append(current_run)
        
        results.append({
            'sequence': f"{station}_{track_id}_{side}",
            'total_frames': len(group),
            'defective_frames': n_defective,
            'isolated_defects': isolated,
            'clustered_defects': clustered,
            'clustering_ratio': clustered / n_defective if n_defective > 0 else 0,
            'num_defect_runs': len(runs),
            'mean_run_length': np.mean(runs) if runs else 0,
            'max_run_length': max(runs) if runs else 0,
            'frame_range': f"{frames[0]:03d}-{frames[-1]:03d}"
        })
    
    results_df = pd.DataFrame(results)
    
    # Summary statistics
    print("TEMPORAL CLUSTERING ANALYSIS")
    print(f"\nSequences analyzed: {len(df.groupby(['Station', 'TrackID', 'Side']))}")
    print(f"Sequences with defects: {len(results_df)}")
    print(f"Sequences without defects: {len(df.groupby(['Station', 'TrackID', 'Side'])) - len(results_df)}")
    
    print(f"\n DEFECT FRAME STATISTICS ")
    print(f"Total defective frames: {results_df['defective_frames'].sum()}")
    print(f"Isolated defects: {results_df['isolated_defects'].sum()} ({results_df['isolated_defects'].sum()/results_df['defective_frames'].sum()*100:.1f}%)")
    print(f"Clustered defects: {results_df['clustered_defects'].sum()} ({results_df['clustered_defects'].sum()/results_df['defective_frames'].sum()*100:.1f}%)")
    
    print(f"\n CLUSTERING METRICS ")
    print(f"Mean clustering ratio per sequence: {results_df['clustering_ratio'].mean():.2f}")
    print(f"Median clustering ratio per sequence: {results_df['clustering_ratio'].median():.2f}")
    
    print(f"\n RUN LENGTH STATISTICS ")
    print(f"Total number of defect runs: {results_df['num_defect_runs'].sum()}")
    print(f"Mean run length across all sequences: {results_df['mean_run_length'].mean():.2f} frames")
    print(f"Median run length: {results_df['mean_run_length'].median():.2f} frames")
    print(f"Longest defect run: {results_df['max_run_length'].max()} frames")
    
    # Show examples
    print(f"\n EXAMPLES ")
    print("\nTop 5 sequences with highest clustering:")
    top_clustered = results_df.nlargest(5, 'clustering_ratio')[['sequence', 'frame_range', 'total_frames', 'defective_frames', 'clustering_ratio', 'mean_run_length']]
    print(top_clustered.to_string(index=False))
    
    print("\nTop 5 sequences with most isolated defects:")
    top_isolated = results_df.nlargest(5, 'isolated_defects')[['sequence', 'frame_range', 'total_frames', 'defective_frames', 'isolated_defects', 'clustered_defects']]
    print(top_isolated.to_string(index=False))
    
    print("\n SEQUENCES WITH LONG DEFECT RUNS ")
    long_runs = results_df[results_df['max_run_length'] >= 5].sort_values('max_run_length', ascending=False)
    if len(long_runs) > 0:
        print(f"\nFound {len(long_runs)} sequences with defect runs ≥ 5 frames:")
        print(long_runs[['sequence', 'frame_range', 'total_frames', 'defective_frames', 'max_run_length', 'num_defect_runs']].to_string(index=False))
    else:
        print("\nNo sequences with defect runs ≥ 5 frames")
    
    print("\n" + "-"*10)
    
    # Save detailed results
    results_df.to_csv("temporal_clustering_analysis.csv", index=False)
    print("\nDetailed results saved to: temporal_clustering_analysis.csv")
    
    return results_df

# Run the analysis
clustering_results = analyze_temporal_clustering(".//sequential_noprep_labeled")

# Additional visualization: distribution of run lengths
print("\n RUN LENGTH DISTRIBUTION ")
run_length_counts = {}
for _, row in clustering_results.iterrows():
    runs_in_seq = int(row['num_defect_runs'])
    if runs_in_seq > 0:
        # Approximate: we only have mean, but good enough for distribution
        approx_length = round(row['mean_run_length'])
        run_length_counts[approx_length] = run_length_counts.get(approx_length, 0) + runs_in_seq

if run_length_counts:
    print(f"\nRun length histogram:")
    for length in sorted(run_length_counts.keys()):
        count = run_length_counts[length]
        bar = "█" * min(50, count)
        print(f"  {length:2d} frames: {bar} ({count})")

TEMPORAL CLUSTERING ANALYSIS

Sequences analyzed: 71
Sequences with defects: 27
Sequences without defects: 44

 DEFECT FRAME STATISTICS 
Total defective frames: 415
Isolated defects: 17 (4.1%)
Clustered defects: 398 (95.9%)

 CLUSTERING METRICS 
Mean clustering ratio per sequence: 0.61
Median clustering ratio per sequence: 1.00

 RUN LENGTH STATISTICS 
Total number of defect runs: 44
Mean run length across all sequences: 8.16 frames
Median run length: 2.00 frames
Longest defect run: 100 frames

 EXAMPLES 

Top 5 sequences with highest clustering:
                        sequence frame_range  total_frames  defective_frames  clustering_ratio  mean_run_length
    11_main_station_11.1_0_right     276-285            10                 2               1.0              2.0
    17_signal_bridge_17.1_0_left     065-074            10                 3               1.0              3.0
    17_signal_bridge_17.1_1_left     065-074            10                 2               1.0              2.0

In [21]:
def extract_station(filename: str):
    name = os.path.splitext(filename)[0]
    m = re.match(r'(.+?\d+\.\d+)', name)
    return m.group(1) if m else None

def extract_label(filename: str):
    name = os.path.splitext(filename)[0]
    m = re.search(r'_(-?\d+)_(left|right)$', name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def extract_side(filename: str):
    name = os.path.splitext(filename)[0]
    m = re.search(r'_(left|right)$', name, flags=re.IGNORECASE)
    return m.group(1).lower() if m else None


def read_folder(root, label_name, nice_label):
    folder = os.path.join(root, label_name)
    rows = []
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        if not os.path.isfile(path):
            continue
        rows.append({
            "Station": extract_station(f),
            "TrackID": extract_label(f),
            "Side": extract_side(f),
            "Class": nice_label   
        })
    return pd.DataFrame(rows).dropna()


def build_report(labeled_root):
    # Read both folders and give them clean class labels
    df_good = read_folder(labeled_root, "sequential_noprep_good", "good")
    df_bad  = read_folder(labeled_root, "sequential_noprep_defects_maskscreated", "defective")

    df = pd.concat([df_good, df_bad], ignore_index=True)

    counts = (
        df.groupby(["Station","TrackID","Side","Class"])
        .size()
        .reset_index(name="Count")
    )

    # Pivot to get good / defective columns
    table = counts.pivot_table(
        index=["Station","TrackID","Side"],
        columns="Class",
        values="Count",
        fill_value=0
    ).reset_index()

    for col in ["good","defective"]:
        if col not in table:
            table[col] = 0

    table["total"] = table["good"] + table["defective"]

    # Sort nicely
    table = table.sort_values(["Station","TrackID","Side"])

    return table

report = build_report("./sequential/sequential_noprep_labeled")

# Show full table 
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)

print(report)

report.to_csv("patch_report.csv", index=False)
print("\nSaved to patch_report.csv")

Class                             Station  TrackID   Side  defective   good  total
0               10_station_suelldorf_10.1        0   left        0.0   10.0   10.0
1               10_station_suelldorf_10.1        0  right        0.0   10.0   10.0
2                    11_main_station_11.1        0   left        0.0   10.0   10.0
3                    11_main_station_11.1        0  right        2.0    8.0   10.0
4                13_station_ohlsdorf_13.1       -1   left        0.0   10.0   10.0
5                13_station_ohlsdorf_13.1       -1  right        0.0   10.0   10.0
6                13_station_ohlsdorf_13.1        0   left        0.0   10.0   10.0
7                13_station_ohlsdorf_13.1        0  right        1.0    9.0   10.0
8                 14_signals_station_14.1       -1   left        0.0   10.0   10.0
9                 14_signals_station_14.1       -1  right        1.0    9.0   10.0
10                14_signals_station_14.1        0   left        1.0    9.0   10.0
11  

# Important Functions to help split data

Goal:  Stratified K-folds split using Hamilton apportionment

In [ ]:
# Data classes
@dataclass(frozen=True)
class PatchRecord:
    path: str
    station: str
    photo_name: str
    x: int
    y: int
    angle: float
    label_id: str
    side: str
    is_defective: bool
    mask_path: Optional[str] = None

@dataclass(frozen=True)
class Sequence:
    station: str
    side: str
    label_id: str
    n_patches: int
    n_defective: int
    n_normal: int
    any_defect: bool
    patch_paths: Tuple[str, ...]
    patch_labels: Tuple[str, ...]   # 'defective' or 'normal'
    mask_paths: Tuple[Optional[str], ...]


# Parsing & indexing

def strip_ext_and_mask(stem: str):
    if '.' in stem:
        stem_noext = stem.rsplit('.', 1)[0]
    else:
        stem_noext = stem
    is_mask = False
    if stem_noext.endswith('_mask'):
        stem_noext = stem_noext[:-5]
        is_mask = True
    return stem_noext, is_mask

def parse_from_end(filename: str) -> Dict[str, str]:
    stem = os.path.basename(filename)
    stem_noext, _ = strip_ext_and_mask(stem)
    parts = stem_noext.split('_')
    if len(parts) < 8:
        raise ValueError(f'Unexpected filename format: {filename}')
    side = parts[-1]
    label_id = parts[-2]
    angle = parts[-3]
    y = parts[-4]
    x = parts[-5]
    if not (parts[-6] == 'patch' and parts[-7] == 'patches'):
        raise ValueError(f"Missing 'patches_patch' tokens near end: {filename}")
    prefix_parts = parts[:-7]
    prefix_joined = '_'.join(prefix_parts)
    m = re.search(r'^(.*)_(\d+_\d+\.\d+)$', prefix_joined)
    if m:
        station = m.group(1)
        photo_name = m.group(2)
    else:
        if len(prefix_parts) < 2:
            raise ValueError(f'Cannot parse station/photo_name: {filename}')
        station = '_'.join(prefix_parts[:-2])
        photo_name = '_'.join(prefix_parts[-2:])
    xi = int(x); yi = int(y); angf = float(angle)
    return {'station': station, 'photo_name': photo_name, 'x': xi, 'y': yi, 'angle': angf, 'label_id': label_id, 'side': side}

def index_directory(root: str) -> Dict[str, str]:
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    out = {}
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            ext = os.path.splitext(fn)[1].lower()
            if ext in exts or ext == '':
                stem_noext, _ = strip_ext_and_mask(fn)
                out[stem_noext] = os.path.join(dirpath, fn)
    return out

def index_masks(root: Optional[str]) -> Dict[str, str]:
    if not root:
        return {}
    exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    out = {}
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            ext = os.path.splitext(fn)[1].lower()
            if ext in exts or ext == '':
                stem_noext, is_mask = strip_ext_and_mask(fn)
                if is_mask:
                    out[stem_noext] = os.path.join(dirpath, fn)
    return out

def scan_dataset(good_dir: str, defective_dir: str, masks_dir: Optional[str]) -> List[PatchRecord]:
    good_index   = index_directory(good_dir) if good_dir else {}
    defect_index = index_directory(defective_dir) if defective_dir else {}
    mask_index   = index_masks(masks_dir) if masks_dir else {}
    records = []
    for _, fpath in good_index.items():
        fpath_abs = os.path.abspath(fpath)
        meta = parse_from_end(os.path.basename(fpath_abs))
        records.append(PatchRecord(path=fpath_abs, is_defective=False, mask_path=None, **meta))
    for stem_key, fpath in defect_index.items():
        fpath_abs = os.path.abspath(fpath)
        meta = parse_from_end(os.path.basename(fpath_abs))
        mask_path = mask_index.get(stem_key, None)
        mask_abs = os.path.abspath(mask_path) if mask_path else None
        records.append(PatchRecord(path=fpath_abs, is_defective=True, mask_path=mask_abs, **meta))
    return records

def build_sequences_mixed(records: List[PatchRecord], assert_lengths=False, allowed_lengths=(10, 100)) -> List[Sequence]:
    buckets = defaultdict(list)
    for r in records:
        key = (r.station, r.side, r.label_id)
        buckets[key].append(r)
    sequences = []
    for (station, side, label_id), items in buckets.items():
        items_sorted = sorted(items, key=lambda x: x.photo_name)
        n = len(items_sorted)
        n_def = sum(1 for it in items_sorted if it.is_defective)
        n_norm = n - n_def
        if assert_lengths and n not in allowed_lengths:
            print(f"[WARN] Sequence {(station, side, label_id)} has {n} patches (expected {allowed_lengths}).")
        sequences.append(Sequence(
            station=station,
            side=side,
            label_id=label_id,
            n_patches=n,
            n_defective=n_def,
            n_normal=n_norm,
            any_defect=(n_def>0),
            patch_paths=tuple(it.path for it in items_sorted),
            patch_labels=tuple('defective' if it.is_defective else 'normal' for it in items_sorted),
            mask_paths=tuple(it.mask_path for it in items_sorted)
        ))
    return sequences


# Balanced K-fold maker (stratified by length & defect ratio) + quotas + greedy placement

def _apportion_counts(n_by_bucket, target_total, rng):
    base = {b:int(v) for b,v in n_by_bucket.items()}
    assigned = sum(base.values())
    remainder = target_total - assigned
    if remainder <= 0:
        fracs = [ (n_by_bucket[b] - base[b], b) for b in n_by_bucket ]
        fracs.sort()
        i = 0
        while assigned > target_total and i < len(fracs):
            b = fracs[i][1]
            if base[b] > 0:
                base[b] -= 1
                assigned -= 1
            i += 1
        return base
    fracs = [ (n_by_bucket[b] - base[b], rng.random(), b) for b in n_by_bucket ]
    fracs.sort(reverse=True)
    i = 0
    while remainder > 0 and i < len(fracs):
        b = fracs[i][2]
        base[b] += 1
        remainder -= 1
        i += 1
    return base

def make_manual_balanced_kfolds(
    sequences: List[Sequence],
    n_splits: int = 5,
    seed: int = 42,
    w_total_patches: float = 0.5,
    w_defective_patches: float = 1.0,
) -> List[List[int]]:
    """
    Stratified-by-load K-fold at sequence level:
      - strata by (length bin, defect ratio bin)
      - Hamilton apportionment -> per-fold quotas per stratum
      - greedy assignment minimizing w_total*total_patches + w_def*defective_patches
      - keeps #sequences per fold ~ equal
    """
    assert n_splits >= 2
    rng = random.Random(seed)
    N = len(sequences)
    idx_all = list(range(N))

    # strata by sequence length and defect ratio bins
    def strata_key(s: Sequence):
        L = s.n_patches
        lbin = 0 if L <= 10 else (1 if L <= 50 else 2)  # tweak if you have other sizes
        ratio = (s.n_defective / L) if L else 0.0
        rbin = 0 if ratio == 0 else (1 if ratio <= 0.10 else 2 if ratio <= 0.50 else 3)
        return (lbin, rbin)

    strata = defaultdict(list)
    for i in idx_all:
        strata[strata_key(sequences[i])].append(i)
    for k in strata:  # shuffle within stratum
        rng.shuffle(strata[k])

    # folds + tallies
    folds = [[] for _ in range(n_splits)]
    fold_total_patches = [0]*n_splits
    fold_def_patches   = [0]*n_splits
    fold_seq_counts    = [0]*n_splits

    def seq_tot_def(i: int):
        s = sequences[i]
        return s.n_patches, s.n_defective

    # target sequence counts per fold (±1)
    target_seq_per_fold = [N//n_splits + (1 if r < (N % n_splits) else 0) for r in range(n_splits)]

    # apportion each stratum and place sequences greedily
    for key, idxs in strata.items():
        m = len(idxs)
        if m == 0: 
            continue
        desired = {f: m / float(n_splits) for f in range(n_splits)}
        alloc = _apportion_counts(desired, m, rng)
        # place larger & more defective first
        todo = []
        for i in idxs:
            tot, d = seq_tot_def(i)
            todo.append((i, tot, d))
        todo.sort(key=lambda t: (t[1], t[2]), reverse=True)

        placed_in_stratum = [0]*n_splits
        for i, tot, d in todo:
            best_fold = None
            best_score = None
            for f in range(n_splits):
                if placed_in_stratum[f] >= alloc.get(f, 0):
                    continue
                if fold_seq_counts[f] >= target_seq_per_fold[f]:
                    continue
                new_tot = fold_total_patches[f] + tot
                new_def = fold_def_patches[f] + d
                score = w_total_patches * new_tot + w_defective_patches * new_def
                if (best_score is None) or (score < best_score):
                    best_score = score
                    best_fold = f

            # relax seq-target if needed
            if best_fold is None:
                for f in range(n_splits):
                    if placed_in_stratum[f] >= alloc.get(f, 0):
                        continue
                    new_tot = fold_total_patches[f] + tot
                    new_def = fold_def_patches[f] + d
                    score = w_total_patches * new_tot + w_defective_patches * new_def
                    if (best_score is None) or (score < best_score):
                        best_score = score
                        best_fold = f

            if best_fold is None:
                # final fallback
                for f in range(n_splits):
                    new_tot = fold_total_patches[f] + tot
                    new_def = fold_def_patches[f] + d
                    score = w_total_patches * new_tot + w_defective_patches * new_def
                    if (best_score is None) or (score < best_score):
                        best_score = score
                        best_fold = f

            folds[best_fold].append(i)
            fold_total_patches[best_fold] += tot
            fold_def_patches[best_fold]   += d
            fold_seq_counts[best_fold]    += 1
            placed_in_stratum[best_fold]  += 1

    # sanity + quick diagnostics
    used = set()
    for f in range(n_splits):
        folds[f] = sorted(folds[f])
        for i in folds[f]:
            if i in used:
                raise RuntimeError("Index assigned to multiple folds")
            used.add(i)
    if len(used) != N:
        missing = [i for i in idx_all if i not in used]
        raise RuntimeError(f"Some indices not assigned: {missing[:10]} ...")

    print("=== Initial greedy-balanced fold loads ===")
    for f in range(n_splits):
        print(f"Fold {f}: #seq={len(folds[f])}, total_patches={fold_total_patches[f]}, defective_patches={fold_def_patches[f]}")
    return folds


# Optional: small swap-based rebalancer to polish the result

def rebalance_folds_by_swaps(
    sequences, folds, 
    w_total_patches=0.5, w_defective_patches=1.0,
    max_passes=3, verbose=True, restart_after_accept=True
):
    """
    Local search that swaps sequences between folds to reduce imbalance of:
      - total patches per fold
      - defective patches per fold
    Robust to mid-iteration swaps (avoids 'list.remove(x): x not in list').
    """
    K = len(folds)
    folds = [list(f) for f in folds]  # make a working copy

    def fold_stats(folds):
        tot = [0]*K; dft = [0]*K
        for k in range(K):
            for i in folds[k]:
                s = sequences[i]
                tot[k] += s.n_patches
                dft[k] += s.n_defective
        return tot, dft

    def objective(tot, dft):
        avg_tot = sum(tot)/K if K else 0.0
        avg_dft = sum(dft)/K if K else 0.0
        return (w_total_patches * sum((x-avg_tot)**2 for x in tot)
              + w_defective_patches * sum((x-avg_dft)**2 for x in dft))

    tot, dft = fold_stats(folds)
    best_obj = objective(tot, dft)
    if verbose:
        print(f"[rebalance] start obj={best_obj:.2f} | tot={tot} | def={dft}")

    passes = 0
    improved_any = True
    while improved_any and passes < max_passes:
        improved_any = False
        passes += 1

        # Try all fold pairs
        for a in range(K):
            for b in range(a+1, K):
                # Snapshot of candidates (largest / most defective first)
                Aa = sorted(folds[a], key=lambda i: (-sequences[i].n_patches, -sequences[i].n_defective))
                Bb = sorted(folds[b], key=lambda i: (-sequences[i].n_patches, -sequences[i].n_defective))

                i_a = 0
                while i_a < len(Aa):
                    ia = Aa[i_a]
                    i_a += 1
                    # Skip if ia moved due to earlier accepted swaps
                    if ia not in folds[a]:
                        continue
                    sa = sequences[ia]

                    i_b = 0
                    accepted_here = False
                    while i_b < len(Bb):
                        ib = Bb[i_b]
                        i_b += 1
                        # Skip if ib moved due to earlier accepted swaps
                        if ib not in folds[b]:
                            continue
                        sb = sequences[ib]

                        # Compute objective delta
                        new_tot_a = tot[a] - sa.n_patches + sb.n_patches
                        new_tot_b = tot[b] - sb.n_patches + sa.n_patches
                        new_dft_a = dft[a] - sa.n_defective + sb.n_defective
                        new_dft_b = dft[b] - sb.n_defective + sa.n_defective

                        # Temporarily set, evaluate, then either accept or revert
                        old_tot_a, old_tot_b = tot[a], tot[b]
                        old_dft_a, old_dft_b = dft[a], dft[b]
                        tot[a], tot[b] = new_tot_a, new_tot_b
                        dft[a], dft[b] = new_dft_a, new_dft_b
                        new_obj = objective(tot, dft)

                        if new_obj + 1e-9 < best_obj:
                            # Accept swap (ensure membership still valid)
                            if ia in folds[a] and ib in folds[b]:
                                folds[a].remove(ia); folds[a].append(ib)
                                folds[b].remove(ib); folds[b].append(ia)
                                best_obj = new_obj
                                improved_any = True
                                accepted_here = True
                                if verbose:
                                    print(f"[rebalance] pass {passes} ↓ obj={best_obj:.2f} by swapping "
                                          f"{ia}(a) ↔ {ib}(b) | tot={tot} def={dft}")
                                # After accept, refresh candidate lists for current pair if requested
                                if restart_after_accept:
                                    Aa = sorted(folds[a], key=lambda i: (-sequences[i].n_patches, -sequences[i].n_defective))
                                    Bb = sorted(folds[b], key=lambda i: (-sequences[i].n_patches, -sequences[i].n_defective))
                                    i_a = 0  # restart outer list
                                    break  # break inner loop to restart with fresh Aa/Bb
                            else:
                                # Membership changed while evaluating; treat as no-op and revert stats
                                tot[a], tot[b] = old_tot_a, old_tot_b
                                dft[a], dft[b] = old_dft_a, old_dft_b
                        else:
                            # Revert stats (no accept)
                            tot[a], tot[b] = old_tot_a, old_tot_b
                            dft[a], dft[b] = old_dft_a, old_dft_b

                    if accepted_here and restart_after_accept:
                        # After refreshing Aa/Bb we restarted outer loop, so break to while re-check
                        continue

        if verbose:
            tot, dft = fold_stats(folds)
            print(f"[rebalance] end pass {passes} | obj={best_obj:.2f} | tot={tot} | def={dft}")

    # Final tidy
    return [sorted(f) for f in folds]


# Export helpers (flat structure + manifest)

def _hash8(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:8]

def _unique_name(dst_dir: Path, src_path: str) -> str:
    base = os.path.basename(src_path)
    candidate = base
    i = 1
    while (dst_dir / candidate).exists():
        stem, ext = os.path.splitext(base)
        candidate = f"{stem}_{i}{ext}"
        i += 1
    return candidate

def _safe_link_or_copy(src: str, dst: Path, prefer_symlink: bool):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src or not os.path.exists(src):
        return False
    if prefer_symlink:
        try:
            if os.name == 'nt':
                try:
                    os.link(src, dst)  # hardlink on Windows
                    return True
                except Exception:
                    pass
            os.symlink(src, dst)
            return True
        except Exception:
            pass
    try:
        shutil.copy2(src, dst)
        return True
    except Exception:
        return False

def expand_to_file_list_mixed(sequences: List[Sequence], indices: List[int]):
    rows = []
    for i in indices:
        s = sequences[i]
        for pth, lab, msk in zip(s.patch_paths, s.patch_labels, s.mask_paths):
            rows.append({
                'filepath': pth,
                'mask_path': msk if lab=='defective' and msk is not None else '',
                'station': s.station,
                'side': s.side,
                'label_id': s.label_id,
                'patch_label': lab,
                'sequence_any_defect': int(s.any_defect),
                'sequence_length': s.n_patches
            })
    return rows

def export_folds_simple(sequences, folds, out_dir: str, prefer_symlink: bool, write_masks: bool, manifest_csv: str):
    out_root = Path(out_dir); out_root.mkdir(parents=True, exist_ok=True)
    all_rows = []
    N = len(sequences); all_idx = set(range(N))
    for fold_id, val_idx in enumerate(folds):
        val_set = set(val_idx)
        train_idx = sorted(list(all_idx - val_set))
        split_rows = {'train': expand_to_file_list_mixed(sequences, train_idx),
                      'val':   expand_to_file_list_mixed(sequences, val_idx)}
        for split_name, rows in split_rows.items():
            for r in rows:
                cls = 'good' if r['patch_label']=='normal' else 'defective'
                src_img = r['filepath']
                fold_dir = out_root / f"fold_{fold_id}"
                dst_dir  = fold_dir / split_name / cls
                dst_name = _unique_name(dst_dir, src_img)
                dst_img  = dst_dir / dst_name
                ok_img = _safe_link_or_copy(src_img, dst_img, prefer_symlink)
                # mask (defective only)
                dst_mask = None; ok_mask = 0
                if write_masks and cls=='defective' and r.get('mask_path'):
                    src_mask = r['mask_path']
                    if src_mask and os.path.exists(src_mask):
                        masks_dir = fold_dir / f"{split_name}_masks" / 'defective'
                        mask_name = _unique_name(masks_dir, src_mask)
                        dst_mask  = masks_dir / mask_name
                        ok_mask   = int(_safe_link_or_copy(src_mask, dst_mask, prefer_symlink))
                all_rows.append({
                    'fold': fold_id, 'split': split_name, 'class': cls,
                    'src_path': src_img, 'dst_relpath': str(dst_img.relative_to(fold_dir)),
                    'mask_src': r.get('mask_path','') if cls=='defective' else '',
                    'mask_dst_relpath': (str(dst_mask.relative_to(fold_dir)) if dst_mask else ''),
                    'copied_img': int(bool(ok_img)), 'copied_mask': ok_mask,
                    'station': r['station'], 'side': r['side'], 'label_id': r['label_id'],
                    'sequence_any_defect': r['sequence_any_defect'], 'sequence_length': r['sequence_length'],
                })
    # manifest
    if manifest_csv is None:
        manifest_csv = str(out_root / "manifest.csv")
    with open(manifest_csv, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['fold','split','class','src_path','dst_relpath','mask_src','mask_dst_relpath',
                      'copied_img','copied_mask','station','side','label_id','sequence_any_defect','sequence_length']
        w = csv.DictWriter(f, fieldnames=fieldnames); w.writeheader()
        for row in all_rows: w.writerow(row)
    print(f"[OK] Wrote flat folds to: {out_root}")
    print(f"[OK] Manifest: {manifest_csv}  (rows={len(all_rows)})")


# Diagnostics & sequence ->fold CSV
def print_fold_summary_table(sequences: List[Sequence], folds: List[List[int]], method_name: str):
    print(f"\n=== {method_name.upper()} METHOD ===")
    total_sequences = len(sequences)
    all_indices = set(range(total_sequences))
    for fold_id, val_indices in enumerate(folds):
        val_set = set(val_indices)
        train_indices = sorted(list(all_indices - val_set))
        train_def = train_good = val_def = val_good = 0
        for idx in train_indices:
            s = sequences[idx]
            train_def  += s.n_defective
            train_good += s.n_normal
        for idx in val_indices:
            s = sequences[idx]
            val_def  += s.n_defective
            val_good += s.n_normal
        train_total = train_def + train_good
        val_total   = val_def + val_good
        print(f"Fold {fold_id}: TRAIN(def={train_def}, good={train_good}, total={train_total}) | "
              f"VAL(def={val_def}, good={val_good}, total={val_total})")

def summarize_from_manifest(manifest_csv_path):
    rows = []
    with open(manifest_csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append(r)
    counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    for r in rows:
        fold  = int(r["fold"]); split = r["split"]; cls = r["class"]
        counts[fold][split][cls] += 1
    print("=== Per-fold class counts from manifest (patch-level) ===")
    for fold in sorted(counts.keys()):
        tr_def = counts[fold]["train"].get("defective", 0)
        tr_good= counts[fold]["train"].get("good", 0)
        va_def = counts[fold]["val"].get("defective", 0)
        va_good= counts[fold]["val"].get("good", 0)
        print(f"Fold {fold}: TRAIN(def={tr_def}, good={tr_good}, total={tr_def+tr_good}) | "
              f"VAL(def={va_def}, good={va_good}, total={va_def+va_good})")

def export_sequence_fold_manifest(sequences, folds, out_csv):
    fold_of = {}
    for f_id, idxs in enumerate(folds):
        for i in idxs:
            if i in fold_of:
                raise RuntimeError(f"Sequence {i} in multiple folds.")
            fold_of[i] = f_id
    missing = [i for i in range(len(sequences)) if i not in fold_of]
    if missing:
        raise RuntimeError(f"{len(missing)} sequences not assigned: {missing[:10]} ...")
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    fields = ["station","side","label_id","n_patches","n_normal","n_defective","any_defect","fold"]
    with open(out_csv, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for i, s in enumerate(sequences):
            w.writerow({"station":s.station,"side":s.side,"label_id":s.label_id,
                        "n_patches":s.n_patches,"n_normal":s.n_normal,"n_defective":s.n_defective,
                        "any_defect":int(s.any_defect),"fold":fold_of[i]})
    print(f"[OK] Sequence->fold manifest: {out_csv}")




## Outer data split 5 folds (keep one out) 

In [ ]:
def create_outer_folds(good_dir,defective_dir,masks_dir,outer_out_dir,outer_k,outer_seed,prefer_symlink=False, write_masks=True):# Scan dataset -> sequences
    print("[INFO] Scanning dataset and building sequences...")
    records   = scan_dataset(good_dir, defective_dir, masks_dir or None)
    sequences = build_sequences_mixed(records, assert_lengths=False)

    print(f"[INFO] Sequences total: {len(sequences)}")
    print(f"[INFO] Any-defect sequences: {sum(int(s.any_defect) for s in sequences)}")
    print(f"[INFO] Sequence lengths: {sorted(set(s.n_patches for s in sequences))}")

    # Make OUTER folds (balanced)
    outer_folds = make_manual_balanced_kfolds(
        sequences, n_splits=outer_k, seed=outer_seed,
        w_total_patches=0.5, w_defective_patches=1.0
    )

    # Optional: polish with a couple of swap passes
    outer_folds = rebalance_folds_by_swaps(
        sequences, outer_folds,
        w_total_patches=0.5, w_defective_patches=1.0,
        max_passes=2, verbose=True
    )

    # Diagnostics (patch-level counts)
    print_fold_summary_table(sequences, outer_folds, "Balanced+Rebalanced (OUTER)")

    # Export flat structure + manifest + sequence->fold CSV
    outer_manifest_csv = str(Path(outer_out_dir)/"manifest_outer.csv")
    export_folds_simple(sequences, outer_folds, outer_out_dir, prefer_symlink, write_masks, outer_manifest_csv)

    outer_seq_manifest = str(Path(outer_out_dir)/"sequence_fold_manifest_outer.csv")
    export_sequence_fold_manifest(sequences, outer_folds, outer_seq_manifest)
    return outer_folds, sequences

## Inner split (4remaining folds to 5)

In [ ]:
def inner_5foldsplit(sequences, outer_folds, test_fold_id,inner_k,inner_seed,inner_out_dir,prefer_symlink=False,write_masks=True):
    # outer_folds                                                               # sequences in each fold
    # test_fold_id      = 0                                                     # which OUTER fold to use as TEST
    # inner_k           = 5                                                     # inner CV folds
    # inner_seed        = 42                                                    # seed for inner CV
    # inner_out_dir     = r"./claheblur_data_folds/inner_5fold_balanced51"      # where to export inner folds
    # prefer_symlink    = False                                                 #  copy files
    # write_masks       = True                                                  #  False for not exporting masks 

    # Fix TEST fold
    test_seq_idx = set(outer_folds[test_fold_id])
    print(f"[INFO] Holding out OUTER fold {test_fold_id} as TEST. Sequences in test: {len(test_seq_idx)}")

    # Build TRAINING POOL (remaining 4 folds)
    train_pool_idx = []
    for f_id, idxs in enumerate(outer_folds):
        if f_id == test_fold_id:
            continue
        train_pool_idx.extend(idxs)
    train_pool_idx = sorted(train_pool_idx)
    print(f"[INFO] Training pool sequences (outer folds except {test_fold_id}): {len(train_pool_idx)}")

    # Create a subset 'sequences' list for the pool (so all helper funcs work as-is)
    subset_sequences = [sequences[i] for i in train_pool_idx]

    # Make INNER folds (balanced) on the subset
    inner_folds_local = make_manual_balanced_kfolds(
        subset_sequences, n_splits=inner_k, seed=inner_seed,
        w_total_patches=0.5, w_defective_patches=1.0
    )

    # Optional: polish with swap rebalancer
    inner_folds_local = rebalance_folds_by_swaps(
        subset_sequences, inner_folds_local,
        w_total_patches=0.5, w_defective_patches=1.0,
        max_passes=2, verbose=True
    )

    # Diagnostics on the INNER folds (subset domain)
    print_fold_summary_table(subset_sequences, inner_folds_local, "Balanced+Rebalanced (INNER)")

    # Export INNER folds (flat dirs + manifest) using the subset
    inner_manifest_csv = str(Path(inner_out_dir) / "manifest_inner.csv")
    export_folds_simple(subset_sequences, inner_folds_local, inner_out_dir,
                        prefer_symlink, write_masks, inner_manifest_csv)

    # Sequence->fold manifest for INNER (subset)
    inner_seq_manifest = str(Path(inner_out_dir) / "sequence_fold_manifest_inner.csv")
    export_sequence_fold_manifest(subset_sequences, inner_folds_local, inner_seq_manifest)

    #(Optional) provide mapping back to GLOBAL sequence indices for records
    #This CSV lists the *global* sequence index and which inner fold it belongs to.
    global_map_csv = str(Path(inner_out_dir) / "inner_fold_global_index_map.csv")
    Path(global_map_csv).parent.mkdir(parents=True, exist_ok=True)
    with open(global_map_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["global_seq_index","inner_fold"])
        for fold_id, local_idxs in enumerate(inner_folds_local):
            for j in local_idxs:
                w.writerow([train_pool_idx[j], fold_id])

    print("\n=SUMMARY =")
    print(f"Test set (outer)          : fold {test_fold_id} (NOT used in tuning)")
    print(f"Inner CV folds exported to: {inner_out_dir}")
    print(f"  - Files manifest        : {inner_manifest_csv}")
    print(f"  - Sequence->fold (subset): {inner_seq_manifest}")
    print(f"  - Global index map      : {global_map_csv}")


# Use the functions to split the data (for each preprocessing split is exactly the same)

## Split to original patches (no preproseccing in backround):

In [ ]:
outer_folds, sequences  = create_outer_folds(good_dir= r"./sequential/sequential_noprep_labeled/sequential_noprep_good",
                  defective_dir= r"./sequential/sequential_noprep_labeled/sequential_noprep_defects_maskscreated",
                  masks_dir= r"./sequential/sequential_noprep_labeled/sequential_defective_masks"  ,
                  outer_out_dir= r"./original_data_folds/outer_5folds_balanced",
                  outer_k=5,
                  outer_seed=123)

#outer folds same for all

inner_5foldsplit(sequences , outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=123,inner_out_dir = r"./original_data_folds/original_inner_folds" ,prefer_symlink=False,write_masks=True)

[INFO] Scanning dataset and building sequences...
[INFO] Sequences total: 71
[INFO] Any-defect sequences: 27
[INFO] Sequence lengths: [10, 100]
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=13, total_patches=310, defective_patches=10
Fold 1: #seq=16, total_patches=430, defective_patches=152
Fold 2: #seq=14, total_patches=320, defective_patches=113
Fold 3: #seq=15, total_patches=330, defective_patches=34
Fold 4: #seq=13, total_patches=220, defective_patches=106
[rebalance] start obj=25060.00 | tot=[310, 430, 320, 330, 220] | def=[10, 152, 113, 34, 106]
[rebalance] pass 1 ↓ obj=15556.00 by swapping 40(a) ↔ 26(b) | tot=[310, 430, 320, 330, 220] def=[98, 64, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12670.00 by swapping 20(a) ↔ 40(b) | tot=[400, 340, 320, 330, 220] def=[95, 67, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12566.00 by swapping 40(a) ↔ 48(b) | tot=[400, 340, 320, 330, 220] def=[93, 69, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12520.00 by swapping 49(a) ↔ 10(b) | tot=[400, 340

## Split to patches with black backround:

In [ ]:
outer_folds, sequences = create_outer_folds(good_dir= r"./sequential/patches_ts_black_prep2/patches_ts_black_prep2_labeled/patches_ts_black_prep2_good",
                  defective_dir= r"./sequential/patches_ts_black_prep2/patches_ts_black_prep2_labeled/patches_ts_black_prep2_defects",
                  masks_dir= r"./sequential/patches_ts_black_prep2/patches_ts_black_prep2_labeled/sequential_defective_masks"  ,
                  outer_out_dir= r"./black_data_folds/outer_5folds_balanced",
                  outer_k=5,
                  outer_seed=123)
inner_5foldsplit(sequences , outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=123,inner_out_dir = r"./black_data_folds/black_inner_folds" ,prefer_symlink=False,write_masks=True)

[INFO] Scanning dataset and building sequences...
[INFO] Sequences total: 71
[INFO] Any-defect sequences: 27
[INFO] Sequence lengths: [10, 100]
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=13, total_patches=310, defective_patches=10
Fold 1: #seq=16, total_patches=430, defective_patches=152
Fold 2: #seq=14, total_patches=320, defective_patches=113
Fold 3: #seq=15, total_patches=330, defective_patches=34
Fold 4: #seq=13, total_patches=220, defective_patches=106
[rebalance] start obj=25060.00 | tot=[310, 430, 320, 330, 220] | def=[10, 152, 113, 34, 106]
[rebalance] pass 1 ↓ obj=15556.00 by swapping 40(a) ↔ 26(b) | tot=[310, 430, 320, 330, 220] def=[98, 64, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12670.00 by swapping 20(a) ↔ 40(b) | tot=[400, 340, 320, 330, 220] def=[95, 67, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12566.00 by swapping 40(a) ↔ 48(b) | tot=[400, 340, 320, 330, 220] def=[93, 69, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12520.00 by swapping 49(a) ↔ 10(b) | tot=[400, 340

## Split to patches with black backround and clahe(applied to track region)

In [ ]:
outer_folds, sequences = create_outer_folds(good_dir= r"./sequential/blackclahe_images/blackclahe_images_labeled/blackclahe_images_good",
                  defective_dir= r"./sequential/blackclahe_images/blackclahe_images_labeled/blackclahe_images_defects",
                  masks_dir= r"./sequential/blackclahe_images/blackclahe_images_labeled/sequential_defective_masks"  ,
                  outer_out_dir= r"./blackclahe_data_folds1/outer_5folds_balanced",
                  outer_k=5,
                  outer_seed=123)


inner_5foldsplit(sequences, outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=123,inner_out_dir = r"./blackclahe_data_folds1/blackclahe_inner_folds" ,prefer_symlink=False,write_masks=True)


[INFO] Scanning dataset and building sequences...
[INFO] Sequences total: 71
[INFO] Any-defect sequences: 27
[INFO] Sequence lengths: [10, 100]
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=13, total_patches=310, defective_patches=10
Fold 1: #seq=16, total_patches=430, defective_patches=152
Fold 2: #seq=14, total_patches=320, defective_patches=113
Fold 3: #seq=15, total_patches=330, defective_patches=34
Fold 4: #seq=13, total_patches=220, defective_patches=106
[rebalance] start obj=25060.00 | tot=[310, 430, 320, 330, 220] | def=[10, 152, 113, 34, 106]
[rebalance] pass 1 ↓ obj=15556.00 by swapping 40(a) ↔ 26(b) | tot=[310, 430, 320, 330, 220] def=[98, 64, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12670.00 by swapping 20(a) ↔ 40(b) | tot=[400, 340, 320, 330, 220] def=[95, 67, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12566.00 by swapping 40(a) ↔ 48(b) | tot=[400, 340, 320, 330, 220] def=[93, 69, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12520.00 by swapping 49(a) ↔ 10(b) | tot=[400, 340

## Split to patches with black backround and clahe applied to track region

In [ ]:
outer_folds, sequences = create_outer_folds(good_dir= r"./sequential/blur_backround_images/blur_backround_images_labeled/blur_backround_images_good",
                  defective_dir= r"./sequential/blur_backround_images/blur_backround_images_labeled/blur_backround_images_defects",
                  masks_dir= r"./sequential/blur_backround_images/blur_backround_images_labeled/sequential_defective_masks"  ,
                  outer_out_dir= r"./blur_data_folds/outer_5folds_balanced",
                  outer_k=5,
                  outer_seed=123)

inner_5foldsplit(sequences, outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=123,inner_out_dir = r"./blur_data_folds/blur_inner_folds" ,prefer_symlink=False,write_masks=True)

[INFO] Scanning dataset and building sequences...
[INFO] Sequences total: 71
[INFO] Any-defect sequences: 27
[INFO] Sequence lengths: [10, 100]
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=13, total_patches=310, defective_patches=10
Fold 1: #seq=16, total_patches=430, defective_patches=152
Fold 2: #seq=14, total_patches=320, defective_patches=113
Fold 3: #seq=15, total_patches=330, defective_patches=34
Fold 4: #seq=13, total_patches=220, defective_patches=106
[rebalance] start obj=25060.00 | tot=[310, 430, 320, 330, 220] | def=[10, 152, 113, 34, 106]
[rebalance] pass 1 ↓ obj=15556.00 by swapping 40(a) ↔ 26(b) | tot=[310, 430, 320, 330, 220] def=[98, 64, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12670.00 by swapping 20(a) ↔ 40(b) | tot=[400, 340, 320, 330, 220] def=[95, 67, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12566.00 by swapping 40(a) ↔ 48(b) | tot=[400, 340, 320, 330, 220] def=[93, 69, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12520.00 by swapping 49(a) ↔ 10(b) | tot=[400, 340

## Split to patches with blur backround and clahe applied to track region

In [ ]:
outer_folds, sequencesB = create_outer_folds(good_dir= r"./sequential/blurclahe_images/blurclahe_images_labeled/blurclahe_images_good",
                  defective_dir= r"./sequential/blurclahe_images/blurclahe_images_labeled/blurclahe_images_defects",
                  masks_dir= r"./sequential/blurclahe_images/blurclahe_images_labeled/sequential_defective_masks"  ,
                  outer_out_dir= r"./blurclahe_data_folds/outer_5folds_balanced",
                  outer_k=5,
                  outer_seed=123)

inner_5foldsplit(sequencesB, outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=123,inner_out_dir = r"./blurclahe_data_folds/blurclahe_inner_seed123" ,prefer_symlink=False,write_masks=True)


[INFO] Scanning dataset and building sequences...
[INFO] Sequences total: 71
[INFO] Any-defect sequences: 27
[INFO] Sequence lengths: [10, 100]
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=13, total_patches=310, defective_patches=10
Fold 1: #seq=16, total_patches=430, defective_patches=152
Fold 2: #seq=14, total_patches=320, defective_patches=113
Fold 3: #seq=15, total_patches=330, defective_patches=34
Fold 4: #seq=13, total_patches=220, defective_patches=106
[rebalance] start obj=25060.00 | tot=[310, 430, 320, 330, 220] | def=[10, 152, 113, 34, 106]
[rebalance] pass 1 ↓ obj=15556.00 by swapping 40(a) ↔ 26(b) | tot=[310, 430, 320, 330, 220] def=[98, 64, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12670.00 by swapping 20(a) ↔ 40(b) | tot=[400, 340, 320, 330, 220] def=[95, 67, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12566.00 by swapping 40(a) ↔ 48(b) | tot=[400, 340, 320, 330, 220] def=[93, 69, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12520.00 by swapping 49(a) ↔ 10(b) | tot=[400, 340

## Split to patches with blur backround and clahe applied to track region and bw everywhere

In [ ]:
outer_folds, sequences = create_outer_folds(good_dir= r"./sequential/blurclahebw_images/blurclahebw_images_labeled/blurclahebw_images_good",
                  defective_dir= r"./sequential/blurclahebw_images/blurclahebw_images_labeled/blurclahebw_images_defects",
                  masks_dir= r"./sequential/blurclahebw_images/blurclahebw_images_labeled/sequential_defective_masks"  ,
                  outer_out_dir= r"./blurclahebw_data_folds/outer_5folds_balanced",
                  outer_k=5,
                  outer_seed=123)

inner_5foldsplit(sequences, outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=123,inner_out_dir = r"./blurclahebw_data_folds/blurclahebw_inner_folds" ,prefer_symlink=False,write_masks=True)


[INFO] Scanning dataset and building sequences...
[INFO] Sequences total: 71
[INFO] Any-defect sequences: 27
[INFO] Sequence lengths: [10, 100]
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=13, total_patches=310, defective_patches=10
Fold 1: #seq=16, total_patches=430, defective_patches=152
Fold 2: #seq=14, total_patches=320, defective_patches=113
Fold 3: #seq=15, total_patches=330, defective_patches=34
Fold 4: #seq=13, total_patches=220, defective_patches=106
[rebalance] start obj=25060.00 | tot=[310, 430, 320, 330, 220] | def=[10, 152, 113, 34, 106]
[rebalance] pass 1 ↓ obj=15556.00 by swapping 40(a) ↔ 26(b) | tot=[310, 430, 320, 330, 220] def=[98, 64, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12670.00 by swapping 20(a) ↔ 40(b) | tot=[400, 340, 320, 330, 220] def=[95, 67, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12566.00 by swapping 40(a) ↔ 48(b) | tot=[400, 340, 320, 330, 220] def=[93, 69, 113, 34, 106]
[rebalance] pass 1 ↓ obj=12520.00 by swapping 49(a) ↔ 10(b) | tot=[400, 340

# For experiment 2, 5 different 5-fold split to best pipeline chosen from experiment 1 (blurclahe)

In [ ]:
inner_5foldsplit(sequencesB,  outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=0,inner_out_dir = r"./blurclahe_data_folds/blurclahe_inner_seed0" ,prefer_symlink=False,write_masks=True)

[INFO] Holding out OUTER fold 0 as TEST. Sequences in test: 13
[INFO] Training pool sequences (outer folds except 0): 58
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=10, total_patches=190, defective_patches=116
Fold 1: #seq=12, total_patches=300, defective_patches=60
Fold 2: #seq=15, total_patches=510, defective_patches=128
Fold 3: #seq=11, total_patches=200, defective_patches=15
Fold 4: #seq=10, total_patches=100, defective_patches=6
[rebalance] start obj=61676.00 | tot=[190, 300, 510, 200, 100] | def=[116, 60, 128, 15, 6]
[rebalance] pass 1 ↓ obj=61166.00 by swapping 57(a) ↔ 20(b) | tot=[190, 300, 510, 200, 100] def=[65, 111, 128, 15, 6]
[rebalance] pass 1 ↓ obj=60230.00 by swapping 56(a) ↔ 26(b) | tot=[280, 210, 510, 200, 100] def=[57, 119, 128, 15, 6]
[rebalance] pass 1 ↓ obj=59108.00 by swapping 20(a) ↔ 57(b) | tot=[280, 210, 510, 200, 100] def=[108, 68, 128, 15, 6]
[rebalance] pass 1 ↓ obj=58956.00 by swapping 16(a) ↔ 48(b) | tot=[280, 210, 510, 200, 100] def=[106, 70,

In [ ]:
inner_5foldsplit(sequencesB,outer_folds, test_fold_id=0,inner_k=5,inner_seed=42,inner_out_dir = r"./blurclahe_data_folds/blurclahe_inner_seed42" ,prefer_symlink=False,write_masks=True)

[INFO] Holding out OUTER fold 0 as TEST. Sequences in test: 13
[INFO] Training pool sequences (outer folds except 0): 58
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=11, total_patches=200, defective_patches=7
Fold 1: #seq=13, total_patches=310, defective_patches=154
Fold 2: #seq=11, total_patches=290, defective_patches=118
Fold 3: #seq=13, total_patches=310, defective_patches=34
Fold 4: #seq=10, total_patches=190, defective_patches=12
[rebalance] start obj=25064.00 | tot=[200, 310, 290, 310, 190] | def=[7, 154, 118, 34, 12]
[rebalance] pass 1 ↓ obj=14944.00 by swapping 27(a) ↔ 21(b) | tot=[200, 310, 290, 310, 190] def=[99, 62, 118, 34, 12]
[rebalance] pass 1 ↓ obj=12940.00 by swapping 48(a) ↔ 27(b) | tot=[290, 220, 290, 310, 190] def=[96, 65, 118, 34, 12]
[rebalance] pass 1 ↓ obj=12880.00 by swapping 13(a) ↔ 11(b) | tot=[290, 220, 290, 310, 190] def=[95, 66, 118, 34, 12]
[rebalance] pass 1 ↓ obj=12824.00 by swapping 6(a) ↔ 0(b) | tot=[290, 220, 290, 310, 190] def=[94, 67, 11

In [ ]:
inner_5foldsplit(sequencesB, outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=79,inner_out_dir = r"./blurclahe_data_folds/blurclahe_inner_seed79" ,prefer_symlink=False,write_masks=True)

[INFO] Holding out OUTER fold 0 as TEST. Sequences in test: 13
[INFO] Training pool sequences (outer folds except 0): 58
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=11, total_patches=290, defective_patches=61
Fold 1: #seq=12, total_patches=210, defective_patches=113
Fold 2: #seq=13, total_patches=310, defective_patches=105
Fold 3: #seq=11, total_patches=200, defective_patches=8
Fold 4: #seq=11, total_patches=290, defective_patches=38
[rebalance] start obj=13098.00 | tot=[290, 210, 310, 200, 290] | def=[61, 113, 105, 8, 38]
[rebalance] pass 1 ↓ obj=12996.00 by swapping 20(a) ↔ 57(b) | tot=[290, 210, 310, 200, 290] def=[112, 62, 105, 8, 38]
[rebalance] pass 1 ↓ obj=12804.00 by swapping 56(a) ↔ 55(b) | tot=[290, 210, 310, 200, 290] def=[110, 64, 105, 8, 38]
[rebalance] pass 1 ↓ obj=12546.00 by swapping 55(a) ↔ 17(b) | tot=[290, 210, 310, 200, 290] def=[107, 67, 105, 8, 38]
[rebalance] pass 1 ↓ obj=12468.00 by swapping 12(a) ↔ 38(b) | tot=[290, 210, 310, 200, 290] def=[106, 68,

In [ ]:
inner_5foldsplit(sequencesB, outer_folds = outer_folds, test_fold_id=0,inner_k=5,inner_seed=26,inner_out_dir = r"./blurclahe_data_folds/blurclahe_inner_seed26" ,prefer_symlink=False,write_masks=True)

[INFO] Holding out OUTER fold 0 as TEST. Sequences in test: 13
[INFO] Training pool sequences (outer folds except 0): 58
=== Initial greedy-balanced fold loads ===
Fold 0: #seq=12, total_patches=210, defective_patches=13
Fold 1: #seq=13, total_patches=400, defective_patches=128
Fold 2: #seq=12, total_patches=300, defective_patches=152
Fold 3: #seq=10, total_patches=190, defective_patches=17
Fold 4: #seq=11, total_patches=200, defective_patches=15
[rebalance] start obj=35146.00 | tot=[210, 400, 300, 190, 200] | def=[13, 128, 152, 17, 15]
[rebalance] pass 1 ↓ obj=32146.00 by swapping 27(a) ↔ 57(b) | tot=[210, 400, 300, 190, 200] def=[113, 28, 152, 17, 15]
[rebalance] pass 1 ↓ obj=31198.00 by swapping 57(a) ↔ 33(b) | tot=[210, 400, 300, 190, 200] def=[34, 107, 152, 17, 15]
[rebalance] pass 1 ↓ obj=26146.00 by swapping 43(a) ↔ 57(b) | tot=[300, 310, 300, 190, 200] def=[128, 13, 152, 17, 15]
[rebalance] pass 1 ↓ obj=23146.00 by swapping 57(a) ↔ 26(b) | tot=[300, 310, 300, 190, 200] def=[28,

# Check for data leakage 

How many sequences went to held-out set 

In [2]:
CV_ROOT = Path("./blurclahe_data_folds/outer_5folds_balanced/fold_0")  # held-out test in val folder there
pattern = re.compile(
    r'^(?P<station>.+?)_'                    # Station
    r'(?P<photo>\d+_\d+\.\d+)_'              # Photo ID (not used here)
    r'patches_patch_'                        # literal
    r'(?P<coords>\d+_\d+_[\d\.]+)_'          # coords & angle (not used)
    r'(?P<track>-?\d+)_'                     # Track ID  <-- used
    r'(?P<side>(?:left|right))'              # Side      <-- used
    r'(?:\..+)?$'
)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
CLASS_DIRS = ["defective", "good"]

def triplet_from_name(name: str):
    m = pattern.match(Path(name).stem)
    if not m:
        return None
    return (m["station"], m["track"], m["side"])  # (station, trackid, side)

def count_sequences(cv_root: Path) -> int:
    seqs = set()
    for split in ["val"]: #how many in held out test
        for cls in CLASS_DIRS:
            base = cv_root / split / cls
            if not base.exists():
                continue
            for p in base.rglob("*"):
                if p.is_file() and p.suffix.lower() in IMG_EXTS:
                    t = triplet_from_name(p.name)
                    if t:
                        seqs.add(t)
    return len(seqs), seqs

n, seqs = count_sequences(CV_ROOT)
print(f"Total unique sequences (station, trackid, side): {n}")



Total unique sequences (station, trackid, side): 13


--> 13 sequences are included in the held-out test set. Thus, 58 sequences left in the trian and val sets (71-13)

In [ ]:
def triplet_from_name(name: str):
    m = pattern.match(Path(name).stem)
    if not m:
        return None
    # UNIQUE SEQUENCE KEY = (station, trackid, side)
    return (m["station"], m["track"], m["side"])

def collect_rows(cv_root: Path) -> pd.DataFrame:
    rows = []
    for split in ["train", "val"]:
        for cls in CLASS_DIRS:
            for p in (cv_root / split / cls).rglob("*"):
                if p.is_file() and p.suffix.lower() in IMG_EXTS:
                    t = triplet_from_name(p.name)
                    if t is None:
                        continue
                    station, track, side = t
                    rows.append({
                        "split": split,
                        "cls": cls,
                        "station": station,
                        "trackid": track,
                        "side": side,
                        "fname": p.name,
                    })
    if not rows:
        raise RuntimeError(f"No images found under {cv_root}")
    return pd.DataFrame(rows)


def check_leakage(CV_ROOT):
    CLASS_DIRS = ["defective", "good"]
    IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}
    EXPECT_NUM_SEQS = 58  # (71-13(in held-out set))

    # filename pattern
    pattern = re.compile(
        r'^(?P<station>.+?)_'                    # Station
        r'(?P<photo>\d+_\d+\.\d+)_'              # Photo ID (not used for group)
        r'patches_patch_'                        
        r'(?P<coords>\d+_\d+_[\d\.]+)_'          
        r'(?P<track>-?\d+)_'                     # Track ID  <-- used
        r'(?P<side>(?:left|right))'              # Side      <-- used
        r'(?:\..+)?$'
    )

    df = collect_rows(CV_ROOT)

    # sets of unique sequences per split
    train_seqs = set(zip(df[df.split=="train"].station, df[df.split=="train"].trackid, df[df.split=="train"].side))
    val_seqs   = set(zip(df[df.split=="val"].station,   df[df.split=="val"].trackid,   df[df.split=="val"].side))
    all_seqs   = train_seqs | val_seqs

    print(f"Total unique sequences (station, trackid, side): {len(all_seqs)}")
    if len(all_seqs) != EXPECT_NUM_SEQS:
        print(f"Expected {EXPECT_NUM_SEQS} sequences, found {len(all_seqs)}. Check folder or regex.")

    # leakage = sequences present in BOTH train and val
    overlap = sorted(train_seqs & val_seqs)

    if overlap:
        print(f"\nLeakage detected: {len(overlap)} sequence triplet(s) appear in BOTH train and val.")
        by_station = defaultdict(list)
        for (station, track, side) in overlap:
            by_station[station].append((track, side))
        for station in sorted(by_station):
            print(f"  station={station}: {sorted(by_station[station])}")
    else:
        print("\nNo (station, trackid, side) leakage between train and val.")

    # sequence assignment table
    tbl = (df.groupby(["station","trackid","side","split"])
            .size()
            .unstack("split", fill_value=0)
            .reset_index()
            .rename(columns={"train":"train_count","val":"val_count"}))
    tbl["in_train"] = tbl["train_count"] > 0
    tbl["in_val"]   = tbl["val_count"] > 0
    tbl["leakage"]  = tbl["in_train"] & tbl["in_val"]

    print("\n== Sequence assignment (first 30 rows) ==")
    print(tbl.sort_values(["leakage","station","trackid","side"], ascending=[False,True,True,True])
            .head(30).to_string(index=False))

    # save full table for reference
    out_csv = CV_ROOT.parent / f"{CV_ROOT.name}_sequence_assignment_by_triplet.csv"
    tbl.to_csv(out_csv, index=False)
    print(f"\nSaved full table to: {out_csv}")


In [5]:
check_leakage(CV_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed26/folds/fold_0") )
check_leakage(CV_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed26/folds/fold_1") )
check_leakage(CV_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed26/folds/fold_2") )
check_leakage(CV_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed26/folds/fold_3") )
check_leakage(CV_ROOT = Path("./blurclahe_data_folds/blurclahe_inner_seed26/folds/fold_4") )

Total unique sequences (station, trackid, side): 58

No (station, trackid, side) leakage between train and val.

== Sequence assignment (first 30 rows) ==
                     station trackid  side  train_count  val_count  in_train  in_val  leakage
   10_station_suelldorf_10.1       0  left           10          0      True   False    False
        11_main_station_11.1       0 right           10          0      True   False    False
    13_station_ohlsdorf_13.1      -1  left           10          0      True   False    False
    13_station_ohlsdorf_13.1      -1 right           10          0      True   False    False
    13_station_ohlsdorf_13.1       0 right           10          0      True   False    False
     14_signals_station_14.1      -1  left           10          0      True   False    False
     14_signals_station_14.1      -1 right           10          0      True   False    False
     14_signals_station_14.1       0  left            0         10     False    True    False

--> Also in every folder with the folds folder, there is a file named sequence_fold_manifest_inner.csv where it is mentioned in which fold each sequence belongs

# Latex: fold with split characteristics

In [4]:

df = pd.read_csv("sequence_fold_manifest_outer.csv", sep=",")
print(df.head())
print(df.columns.tolist())

print(df.columns.tolist())
summary = (
    df.groupby("fold")
      .agg(
          Sequences=("station", "count"),
          Total_Patches=("n_patches", "sum"),
          Defective_Patches=("n_defective", "sum"),
          Stations=("station", pd.Series.nunique)
      )
      .reset_index()
)

summary["Defect_%"] = 100 * summary["Defective_Patches"] / summary["Total_Patches"]


total_row = pd.DataFrame({
    "fold": ["Total"],
    "Sequences": [summary["Sequences"].sum()],
    "Total_Patches": [summary["Total_Patches"].sum()],
    "Defective_Patches": [summary["Defective_Patches"].sum()],
    "Stations": ["--"],
    "Defect_%": [100 * summary["Defective_Patches"].sum() / summary["Total_Patches"].sum()]
})

summary = pd.concat([summary, total_row], ignore_index=True)


latex_table = summary.to_latex(
    index=False,
    header=[
        "Fold",
        "Sequences",
        "Total Patches",
        "Defective Patches",
        "Stations",
        "Defect \\%"
    ],
    caption=(
        "Composition of the selected 5-fold cross-validation split (seed 79) "
        "used in Experiments 3--5. The stratification algorithm balanced sequence count, "
        "patch distribution, defect ratio, and station diversity across folds."
    ),
    label="tab:exp2_seed79_folds",
    escape=False,
    column_format="@{}cccccc@{}"
)


print(latex_table)


                     station   side  label_id  n_patches  n_normal  \
0  10_station_suelldorf_10.1   left         0         10        10   
1  10_station_suelldorf_10.1  right         0         10        10   
2       11_main_station_11.1   left         0         10        10   
3       11_main_station_11.1  right         0         10         8   
4   13_station_ohlsdorf_13.1  right        -1         10        10   

   n_defective  any_defect  fold  
0            0           0     4  
1            0           0     0  
2            0           0     0  
3            2           1     3  
4            0           0     4  
['station', 'side', 'label_id', 'n_patches', 'n_normal', 'n_defective', 'any_defect', 'fold']
['station', 'side', 'label_id', 'n_patches', 'n_normal', 'n_defective', 'any_defect', 'fold']
\begin{table}
\caption{Composition of the selected 5-fold cross-validation split (seed 79) used in Experiments 3--5. The stratification algorithm balanced sequence count, patch dist